In [ ]:
%pip install pandas matplotlib scikit-learn

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

from pandas.plotting import scatter_matrix
from sklearn.preprocessing import PowerTransformer
from sklearn.svm import LinearSVC

from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

In [ ]:
# load the data and take a peek
df = pd.read_csv("stroke_data.csv")
df.head()

In [ ]:
df.info()

In [ ]:
# select 30% to set aside, stratified by stroke
p = 0.3
# Super important to seed your random number generator for reproducibility
rng = np.random.default_rng(42) # yes, this is an over-used random number seed
n_test = int(p * len(df))
is_stroke = df["Stroke"] == "Y"
not_stroke = df["Stroke"] == "N"
n_stroke = int(p * is_stroke.sum())
test_ids = np.hstack([rng.choice(np.where(is_stroke)[0], n_stroke),
                     rng.choice(np.where(~is_stroke)[0], n_test - n_stroke)])
train_ids = np.delete(np.arange(len(df)), test_ids)
test = df.iloc[test_ids]
train = df.iloc[train_ids]

# re-define the is_stroke and not_stroke to relate to the training set
# Also cast to simple list to avoid pandas keyindex weirdness after pipeline
is_stroke = list(train["Stroke"] == "Y")
not_stroke = list(train["Stroke"] == "N")

In [ ]:
# Verify the splits
print(pd.DataFrame([train.value_counts("Stroke"), test.value_counts("Stroke")], ["Train", "Test"]))

In [ ]:
# explore the training data
train.info()

In [ ]:
# Group the imaging predictors and risk factors
VC_preds = ["CALCVol", "CALCVolProp", "MATXVol", "MATXVolProp", "LRNCVol", 
    "LRNCVolProp", "MaxCALCArea", "MaxCALCAreaProp", "MaxDilationByArea", 
    "MaxMATXArea", "MaxMATXAreaProp", "MaxLRNCArea", "MaxLRNCAreaProp", 
    "MaxMaxWallThickness", "MaxRemodelingRatio", "MaxStenosisByArea", 
    "MaxWallArea", "WallVol", "MaxStenosisByDiameter"]
risk_preds = ["age", "sex", "SmokingHistory", "AtrialFibrillation", "CoronaryArteryDisease", 
    "DiabetesHistory", "HypercholesterolemiaHistory", "HypertensionHistory"]

In [ ]:
# Look at the correlation coefficients of the imaging predictors
im_corr = train[VC_preds].corr().values
plt.figure(figsize=(10,8))
plt.imshow(im_corr)
plt.colorbar()
plt.xticks(range(len(VC_preds)), VC_preds, rotation=90)
plt.yticks(range(len(VC_preds)), VC_preds)
plt.show()

In [ ]:
# show that corrcoef only represents linear relationships
test_x = np.arange(-100,100)
test_y = test_x * test_x
plt.scatter(test_x, test_y)
np.corrcoef(test_x, test_y)

In [ ]:
# print out features that are off-diagonal but have high correlation
corr_thresh  = 0.9
exclude = []
scatter_check = []
for i in range(len(VC_preds)):
    for j in range(len(VC_preds)):
        if i < j and abs(im_corr[i, j]) > corr_thresh:
            print(f"{VC_preds[i]} is correlated with {VC_preds[j]}")
            exclude.append(VC_preds[i])
            scatter_check += [VC_preds[i], VC_preds[j]]
            
# filter the VC_preds list
preds = [p for p in VC_preds if p not in exclude]

In [ ]:
# look at a scatter matrix for the excluded features
scatter_matrix(train[scatter_check])#, diagonal='kde')
plt.show()

In [ ]:
# look at the MaxLRNCArea
train[is_stroke]["MaxLRNCArea"].hist(bins=15, alpha=0.5)
train[not_stroke]["MaxLRNCArea"].hist(bins=15, alpha=0.5)
plt.legend(["Stroke == Y", "Stroke == N"])
plt.title("Maximum cross-sectional area of lipid-rich necrotic core")

In [ ]:
# transfrom MaxLRNCArea and re-do the histogram
t = train["MaxLRNCArea"].apply(lambda x: np.log(x))
t[is_stroke].hist(bins=15, alpha=0.5)
t[not_stroke].hist(bins=15, alpha=0.5)
plt.legend(["Stroke == Y", "Stroke == N"])
plt.title("Log of maximum cross-sectional area of lipid-rich necrotic core")

In [ ]:
# look at the histograms of all the predictors
train[preds].hist(figsize=(12,12))
plt.show()

In [ ]:
# Normalize, then look at the histogram again
transformer = PowerTransformer().set_output(transform="pandas")

# always fit to the training data!
train_scaled = transformer.fit_transform(train[preds])

In [ ]:
train_scaled.hist(figsize=(12,12))

In [ ]:
# Any obvious relationships to stroke status?
N = int(np.ceil(len(preds) ** 0.5))
fig,ax = plt.subplots(N,N,figsize=(12,14))
for p, pred in enumerate(preds):
    # figure out the row/col
    row = p // N
    col = p % N
    ax[row][col].boxplot([train_scaled[pred][is_stroke], train_scaled[pred][not_stroke]], tick_labels=["Stroke", "No Stroke"])
    ax[row][col].set_title(pred)

In [ ]:
# Let's try a support vector machine
classifier = LinearSVC()
classifier.fit(X=train_scaled, y=is_stroke)

In [ ]:
# See how well it worked on the training data
train_pred = classifier.predict(train_scaled)
cm = confusion_matrix(is_stroke, train_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=["No Stroke", "Stroke"])
disp.plot()

In [ ]:
# How does it do on our held-back test set?
# BAD BAD BAD
# Note: skipped a bunch of steps, we shouldn't be looking at test yet!
test_pred = classifier.predict(transformer.transform(test[preds]))
cm = confusion_matrix(test["Stroke"] == "Y", test_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=["No Stroke", "Stroke"])
disp.plot()